In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torchvision import models
from pathlib import Path
from PIL import Image

In [26]:
class CircleDataset(Dataset):
    """
    Dataset for circles.
    - df: Pandas DataFrame
    - return_label: Return labels as a tuple (x, y) otherweise return (x, image_id)
    - augment: Apply augmentation (should only be enabled for training)
    """
    def __init__(self, df: pd.DataFrame, img_root: Path, return_label: bool, augment = False):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root
        self.return_label = return_label
        
        from torchvision.transforms import v2
        
        # Задайте размер, который ожидает ваша модель (например, 224)
        IMG_SIZE = 224 

        if augment:
            self.transforms = v2.Compose([
                v2.Resize((IMG_SIZE, IMG_SIZE)), # ОБЯЗАТЕЛЬНО: приводим к одному размеру
                v2.RandAugment(num_ops=2, magnitude=9),
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transforms = v2.Compose([
                v2.Resize((IMG_SIZE, IMG_SIZE)), # Для валидации тоже нужен Resize
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])


    def __len__(self):
        return len(self.df)

    def __getitem__(self, i: int):
        row = self.df.iloc[i]
        image_id = str(row["image_id"])
        rel_path = str(row["image_path"])
        
        img_path = '/kaggle/input/competitions/icdar-2026-circleid-pen-classification' + '/' + rel_path

        img = Image.open(img_path).convert("RGB")
        x = self.transforms(img)
        
        if self.return_label:
            y = int(row["pen_id"])
            y -= 1
            return x, y
        else:
            return x, image_id

In [27]:
BATCH_SIZE = 32
IMG_SIZE = 224
model = model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
# Обращаемся к третьему элементу (индекс 2), где находится Linear слой
num_f = model.classifier[2].in_features

# При замене слоя также цельтесь в индекс 2
model.classifier[2] = torch.nn.Linear(num_f, 8)

#model.load_state_dict(torch.load('/kaggle/working/model_pen_efficientnet_b2.pth', weights_only=True))
device = 'cuda'
model.to(device)
dataset_dir = '/kaggle/input/competitions/icdar-2026-circleid-pen-classification/images'
train_df = pd.read_csv('/kaggle/input/competitions/icdar-2026-circleid-pen-classification/train.csv')
train_ds = CircleDataset(train_df, img_root=dataset_dir, return_label=True, augment=False)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(),lr=0.001)

In [13]:
def train_model(model,criterion,optimizer):
    num_epochs = 8
    for epoch in range(num_epochs):
        i =0
        run_loss= 0.0
        for inp,label in train_loader:
            inp,lab = inp.to(device), label.to(device)
            optimizer.zero_grad()
            out = model(inp)
            loss = criterion(out,lab)
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            i += 1
            if i % 200 == 0:
                print('loss',loss)
        print(f'Epoch [{epoch + 1}], Loss: {run_loss / len(train_loader):.4f}')

In [28]:
train_model(model,criterion,optimizer)

loss tensor(2.1555, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.1768, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.0935, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [1], Loss: 2.1244
loss tensor(2.1331, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.1471, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.0408, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [2], Loss: 2.1254
loss tensor(2.2954, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.0469, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.1452, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [3], Loss: 2.1253
loss tensor(2.2180, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.0630, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.1434, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [4], Loss: 2.1248
loss tensor(2.1169, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(2.1221, device='cuda:0', grad_fn=<NllLossBackwa

In [29]:
torch.save(model.state_dict(), 'pen_convnext.pth')

In [30]:

test_df = pd.read_csv('/kaggle/input/competitions/icdar-2026-circleid-pen-classification/test.csv')
test_ds = CircleDataset1(test_df, img_root=dataset_dir, return_label=False, augment=False)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)



In [31]:
@torch.no_grad()
def predict(model, loader, device, idx_map):
    """Predict for test.csv and format rows for Kaggle submission."""
    model.eval()
    out = []

    for x, image_ids in tqdm(loader, desc="Predict", unit="batch"):
        x = x.to(device, non_blocking=True)
        logits = model(x)


        pred_indices = logits.argmax(dim=1).cpu().numpy()
        for img_id, pi in zip(image_ids, pred_indices):
            out.append((img_id, int(idx_map[int(pi)])))


    return out

In [32]:
def generate_label_maps(train_df: pd.DataFrame):
    """Build label - index maps from the training data."""
 
    labels = sorted(train_df["pen_id"].astype(str).unique().tolist())


    label_map = {label: i for i, label in enumerate(labels)}
    index_map = {i: label for label, i in label_map.items()}

    return label_map, index_map

In [33]:
from tqdm import tqdm

In [34]:
model_state = torch.load('/kaggle/working/pen_convnext.pth', map_location=device)
label_map, idx_map = generate_label_maps(train_df)
predictions = predict(model, test_loader, device, idx_map)

Predict: 100%|██████████| 185/185 [01:03<00:00,  2.92batch/s]


In [35]:
sub = pd.DataFrame(predictions, columns=["image_id", "pen_id"])
out_name = "submission_pen(convnext)2.csv"

sub.to_csv(out_name, index=False)
print(f"Wrote: {out_name}")

Wrote: submission_pen(convnext)2.csv


ансамбль CNN